# Diabetic AI — Knowledge Distillation Training
**Model:** Qwen2.5-3B-Instruct → fine-tuned for Pattern Detection + Recommendation  
**Framework:** Unsloth + TRL SFTTrainer + LoRA  
**Hardware:** Kaggle 2×T4 (32 GB VRAM total)

### Notebook flow
1. Install dependencies  
2. Load model + tokenizer (Unsloth)  
3. Attach LoRA adapters  
4. Load & inspect datasets  
5. Configure SFTTrainer  
6. Train  
7. Evaluate (JSON validity + field accuracy)  
8. Save adapter + push to Hub (optional)

## 0 · Kaggle setup — add your data files first
Upload `train.jsonl`, `val.jsonl`, `test.jsonl` as a Kaggle Dataset, then attach it to this notebook.
The cells below expect them at `/kaggle/input/diabetic-ai-data/`.
Rename the constant below if your dataset slug differs.

In [1]:
# ── Path config — edit only this if your dataset slug differs ──────
DATA_DIR   = "/kaggle/input/datasets/belkaidyacine/diabetic-ai-data"
OUTPUT_DIR = "/kaggle/working/diabetic-ai-lora"
HUB_REPO   = None   # e.g. "your-username/diabetic-ai-qwen25-3b" — set to push to Hub

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

Paths OK


## 1 · Install dependencies

In [2]:
# Unsloth Kaggle-optimised wheel — installs fast, no CUDA recompile needed
# This exact install sequence is required; order matters.
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.returncode

run("pip install -q unsloth")
run("pip install -q --upgrade trl transformers accelerate datasets peft")
run("pip install -q bitsandbytes")

print("✅ Dependencies installed")

✅ Dependencies installed


## 2 · Load model + tokenizer

In [3]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 4096   # covers all patient log inputs comfortably
DTYPE       = None   # auto-detect: bf16 on T4
LOAD_IN_4BIT = False # we have 32 GB — use full LoRA, not QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

print(f"Model loaded | dtype: {model.dtype} | device: {model.device}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.9.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded | dtype: torch.float16 | device: cuda:0
Parameters  : 3.09B


## 3 · Attach LoRA adapters

In [4]:
# ── LoRA config rationale ─────────────────────────────────────────
# r=16      : rank-16 is the sweet spot for 2K-sample fine-tunes;
#             rank-8 undertfits structured JSON tasks, rank-32 overfits.
# alpha=32  : alpha = 2×r is standard; controls effective learning rate
#             of the adapter without touching the base LR.
# dropout=0 : Unsloth recommendation for speed; dataset is clean enough.
# target modules: all attention + FFN projections — needed for JSON
#             structure learning (not just q/v like classic LoRA).

model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    lora_alpha         = 32,
    lora_dropout       = 0,
    target_modules     = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias               = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimised version
    random_state       = 42,
    use_rslora         = False,
    loftq_config       = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable/1e6:.1f}M / {total/1e9:.2f}B  ({100*trainable/total:.2f}%)")

[transformers] Unsloth 2026.5.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable params : 29.9M / 3.12B  (0.96%)


## 4 · Load datasets

In [5]:
import json
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_raw   = load_jsonl(f"{DATA_DIR}/val.jsonl")
test_raw  = load_jsonl(f"{DATA_DIR}/test.jsonl")

print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")

# ── Apply chat template ───────────────────────────────────────────
# SFTTrainer with Unsloth expects a flat 'text' field containing the
# fully formatted prompt+response string. We use the tokenizer's
# built-in chat template so the model sees its native format.

def format_record(record):
    """Convert messages list → single formatted string via chat template."""
    text = tokenizer.apply_chat_template(
        record["messages"],
        tokenize          = False,
        add_generation_prompt = False,  # False = include the assistant turn
    )
    return {"text": text}

train_ds = Dataset.from_list([format_record(r) for r in train_raw])
val_ds   = Dataset.from_list([format_record(r) for r in val_raw])
test_ds  = Dataset.from_list([format_record(r) for r in test_raw])

print(f"\nSample formatted text (first 500 chars):")
print(train_ds[0]["text"][:600])

Train: 1818 | Val: 100 | Test: 100

Sample formatted text (first 500 chars):
<|im_start|>system
You are an expert AI endocrinology assistant. Analyze the patient's demographics and timeline logs to identify physiological glycemic patterns. You must respond STRICTLY with a valid JSON object containing 'feature', 'patterns', and 'message' keys.<|im_end|>
<|im_start|>user
Patient Profile:
- Age: 24
- Sex: M
- Diabetes Type: T1
- Weight: 107 kg

Daily Logs:
D1 09:00 G=138 | 09:30 M=cereal C=50 I=6R | 11:15 G=250 | 13:00 G=145 | 18:30 G=125 | 22:30 I=28L
D2 08:15 G=130 | 08:45 M=pancake C=55 I=7R | 10:30 G=260 | 12:30 G=140 | 18:00 G=120 | 22:00 I=28L
D3 09:30 G=142 | 10:00


In [6]:
# ── Token length audit ────────────────────────────────────────────
# Confirm nothing exceeds MAX_SEQ_LEN (would be silently truncated)

lengths = [
    len(tokenizer.encode(r["text"], add_special_tokens=False))
    for r in train_ds
]

import statistics
print(f"Token length stats (train):")
print(f"  Min    : {min(lengths)}")
print(f"  Max    : {max(lengths)}")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")
print(f"  > 4096 : {sum(1 for l in lengths if l > 4096)} records  ← should be 0")

Token length stats (train):
  Min    : 660
  Max    : 3082
  Mean   : 1347
  Median : 1304
  > 4096 : 0 records  ← should be 0


## 5 · Configure SFTTrainer

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# ── Hyperparameter rationale ──────────────────────────────────────
# per_device_train_batch_size=2 + gradient_accumulation=8
#   → effective batch size = 2 × 2 GPUs × 8 = 32
#   → 32 is the standard for instruction fine-tuning at this scale
#
# learning_rate=2e-4  : standard for LoRA r=16; higher than full FT
#   because adapters start from zero
#
# warmup_ratio=0.05   : 5% warmup → ~270 steps on 1818 samples
#
# lr_scheduler=cosine : decays smoothly, avoids loss spike at end
#
# num_train_epochs=3  : validated choice for ~1800 samples;
#   enough passes to learn JSON structure without memorising
#
# eval_strategy=steps, eval_steps=50 : evaluate on val set every
#   50 steps to catch overfitting early
#
# save_strategy=steps, save_steps=50, load_best_model_at_end=True
#   → keeps the checkpoint with best val loss automatically
training_args = SFTConfig(
    # ── Output ──────────────────────────────────────────────────────
    output_dir                  = OUTPUT_DIR,

    # ── Batch & accumulation ────────────────────────────────────────
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 8,

    # ── Learning rate ───────────────────────────────────────────────
    learning_rate               = 2e-4,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    weight_decay                = 0.01,

    # ── Epochs & steps ──────────────────────────────────────────────
    num_train_epochs            = 3,
    max_steps                   = -1,

    # ── Evaluation & checkpointing ──────────────────────────────────
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    # ── Precision & performance ─────────────────────────────────────
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    dataloader_num_workers      = 2,

    # ── Logging ─────────────────────────────────────────────────────
    logging_steps               = 10,
    logging_first_step          = True,
    report_to                   = "none",

    # ── SFT-specific ────────────────────────────────────────────────
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LEN,
    dataset_num_proc            = 2,
    packing                     = False,

    # ── FIX: disable padding_free (Unsloth sets it True by default) ─
    padding_free                = False,   # ← ADD THIS

    seed                        = 42,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    args            = training_args,
)

print("Trainer configured ✅")
print(f"Steps per epoch : {len(trainer.get_train_dataloader())}")
print(f"Total steps     : {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto (3 epochs)'}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1818 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Trainer configured ✅
Steps per epoch : 909
Total steps     : auto (3 epochs)


## 6 · Train

In [8]:
# ── VRAM snapshot before training ────────────────────────────────
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | VRAM: {max_memory} GB | Reserved before train: {start_gpu_memory} GB")

# ── Train ─────────────────────────────────────────────────────────
trainer_stats = trainer.train()

# ── VRAM snapshot after training ──────────────────────────────────
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nPeak VRAM used   : {used_memory} GB")
print(f"Training runtime : {trainer_stats.metrics['train_runtime']:.0f}s  "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Samples/second   : {trainer_stats.metrics['train_samples_per_second']:.2f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4 | VRAM: 14.563 GB | Reserved before train: 5.926 GB


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,818 | Num Epochs = 3 | Total steps = 342
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.413958,0.411277
100,0.316855,0.304105
150,0.249343,0.263247
200,0.232350,0.234001
250,0.186372,0.222515
300,0.182432,0.214191
342,0.177633,0.212999


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


Peak VRAM used   : 10.447 GB
Training runtime : 9619s  (160.3 min)
Samples/second   : 0.57


## 7 · Evaluate — JSON validity + field accuracy

In [9]:
import re
from collections import Counter, defaultdict

# Switch to inference mode
FastLanguageModel.for_inference(model)

REQUIRED_KEYS = {
    "PATTERNS":        {"feature", "message", "patterns"},
    "RECOMMENDATIONS": {"feature", "message", "recommendations"},
}

def build_prompt(record):
    """Build the prompt-only string (no assistant turn) for generation."""
    messages = record["messages"][:2]  # system + user only
    return tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,   # True = adds the assistant turn opener
    )

def detect_task(record):
    for m in record["messages"]:
        if m["role"] == "system":
            if "patterns" in m["content"]:
                return "PATTERNS"
            if "recommendations" in m["content"]:
                return "RECOMMENDATIONS"
    return "UNKNOWN"

def extract_json(text):
    """Try to extract a JSON object from generated text."""
    # First try direct parse
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    # Try extracting first {...} block
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None

def get_ground_truth_json(record):
    for m in record["messages"]:
        if m["role"] == "assistant":
            try:
                return json.loads(m["content"])
            except Exception:
                return None
    return None

def evaluate(records, split_name="test", max_samples=None):
    """
    Evaluate the model on a list of records.
    Returns a dict of metrics.
    """
    if max_samples:
        records = records[:max_samples]

    metrics = {
        "total":            0,
        "json_valid":       0,
        "feature_correct":  0,
        "keys_complete":    0,
        "array_nonempty":   0,
        "message_nonempty": 0,
        "by_task":          defaultdict(lambda: {"total": 0, "json_valid": 0, "keys_complete": 0}),
    }

    for i, record in enumerate(records):
        task   = detect_task(record)
        prompt = build_prompt(record)

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens      = 600,
                temperature         = 0.1,    # near-deterministic for eval
                do_sample           = True,
                repetition_penalty  = 1.1,
                pad_token_id        = tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens
        input_len  = inputs["input_ids"].shape[1]
        generated  = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        parsed     = extract_json(generated)

        metrics["total"] += 1
        metrics["by_task"][task]["total"] += 1

        if parsed is None:
            if i < 5:   # show first few failures
                print(f"  ⚠️  Sample {i} [{task}] — invalid JSON:")
                print(f"     {generated[:200]}")
            continue

        metrics["json_valid"] += 1
        metrics["by_task"][task]["json_valid"] += 1

        # Feature tag
        if parsed.get("feature") == task:
            metrics["feature_correct"] += 1

        # Required keys
        required  = REQUIRED_KEYS.get(task, set())
        all_keys  = required.issubset(set(parsed.keys()))
        if all_keys:
            metrics["keys_complete"] += 1
            metrics["by_task"][task]["keys_complete"] += 1

        # Non-empty array
        arr_key = "patterns" if task == "PATTERNS" else "recommendations"
        if isinstance(parsed.get(arr_key), list) and len(parsed[arr_key]) > 0:
            metrics["array_nonempty"] += 1

        # Non-empty message
        if isinstance(parsed.get("message"), str) and len(parsed["message"].strip()) > 10:
            metrics["message_nonempty"] += 1

    n = metrics["total"]
    print(f"\n{'='*55}")
    print(f" EVALUATION RESULTS — {split_name.upper()} SET ({n} samples)")
    print(f"{'='*55}")
    print(f"  JSON valid        : {metrics['json_valid']}/{n}  ({100*metrics['json_valid']/n:.1f}%)")
    print(f"  Feature correct   : {metrics['feature_correct']}/{n}  ({100*metrics['feature_correct']/n:.1f}%)")
    print(f"  Keys complete     : {metrics['keys_complete']}/{n}  ({100*metrics['keys_complete']/n:.1f}%)")
    print(f"  Array non-empty   : {metrics['array_nonempty']}/{n}  ({100*metrics['array_nonempty']/n:.1f}%)")
    print(f"  Message non-empty : {metrics['message_nonempty']}/{n}  ({100*metrics['message_nonempty']/n:.1f}%)")

    print(f"\n  Per-task breakdown:")
    for task, t in metrics["by_task"].items():
        tn = t["total"]
        jv = t["json_valid"]
        kc = t["keys_complete"]
        print(f"    [{task}]  total={tn}  json_valid={jv}({100*jv/tn:.0f}%)  keys_complete={kc}({100*kc/tn:.0f}%)")

    return metrics


print("Evaluator defined ✅")

Evaluator defined ✅


In [10]:
# ── Run evaluation on full test set ──────────────────────────────
# Note: this generates one sample at a time (no batching) for
# simplicity and exact metric tracking. Takes ~5-10 min on 100 samples.

test_metrics = evaluate(test_raw, split_name="test")

[transformers] Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local


 EVALUATION RESULTS — TEST SET (100 samples)
  JSON valid        : 99/100  (99.0%)
  Feature correct   : 99/100  (99.0%)
  Keys complete     : 99/100  (99.0%)
  Array non-empty   : 99/100  (99.0%)
  Message non-empty : 99/100  (99.0%)

  Per-task breakdown:
    [PATTERNS]  total=57  json_valid=57(100%)  keys_complete=57(100%)
    [RECOMMENDATIONS]  total=43  json_valid=42(98%)  keys_complete=42(98%)


In [11]:
# ── Quick qualitative check — 3 samples per task ─────────────────
import textwrap

def qualitative_check(records, task_filter, n=3):
    samples = [r for r in records if detect_task(r) == task_filter][:n]
    print(f"\n{'━'*60}")
    print(f" QUALITATIVE CHECK — {task_filter} ({n} samples)")
    print(f"{'━'*60}")

    for i, record in enumerate(samples):
        prompt  = build_prompt(record)
        inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens     = 600,
                temperature        = 0.1,
                do_sample          = True,
                repetition_penalty = 1.1,
                pad_token_id       = tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        parsed    = extract_json(generated)
        ground    = get_ground_truth_json(record)

        print(f"\n── Sample {i+1} ──")

        if task_filter == "PATTERNS" and parsed and ground:
            pred_names  = [p.get("name") for p in parsed.get("patterns", [])]
            true_names  = [p.get("name") for p in ground.get("patterns", [])]
            print(f"  Predicted patterns : {pred_names}")
            print(f"  Ground truth       : {true_names}")
            print(f"  Model message      : {parsed.get('message', '')[:150]}")

        elif task_filter == "RECOMMENDATIONS" and parsed and ground:
            pred_actions = [r.get("action", "")[:80] for r in parsed.get("recommendations", [])]
            true_actions = [r.get("action", "")[:80] for r in ground.get("recommendations", [])]
            print(f"  Predicted actions  : {pred_actions}")
            print(f"  Ground truth       : {true_actions}")
            print(f"  Priority dist      : {[r.get('priority') for r in parsed.get('recommendations', [])]}")
        else:
            print(f"  ⚠️  Could not parse output: {generated[:300]}")

qualitative_check(test_raw, "PATTERNS",        n=3)
qualitative_check(test_raw, "RECOMMENDATIONS", n=3)

[transformers] Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 QUALITATIVE CHECK — PATTERNS (3 samples)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[transformers] Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



── Sample 1 ──
  Predicted patterns : ['Exercise-Induced Drop', 'Post-Lunch Hypoglycemia']
  Ground truth       : ['Post-Lunch Hypoglycemia', 'Exercise-Induced Drop']
  Model message      : This 55-year-old T2 female (111kg) demonstrates two distinct daily patterns: excellent afternoon stability after exercise, and consistent post-lunch h


[transformers] Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


── Sample 3 ──
  Predicted patterns : ['Morning Breakfast Spike', 'Afternoon Stability']
  Ground truth       : ['Morning Breakfast Spike']
  Model message      : Elderly lean patient shows breakfast spike (218-240) and stable afternoons (138-140). Reduce breakfast carbs or add post-meal walk.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 QUALITATIVE CHECK — RECOMMENDATIONS (3 samples)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[transformers] Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



── Sample 1 ──
  Predicted actions  : ['Add Metformin XR at bedtime for dawn phenomenon (fasting 122-128) AND implement ']
  Ground truth       : ['Add metformin XR at bedtime', 'Limit high-fat lunches to twice weekly']
  Priority dist      : ['high']


[transformers] Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



── Sample 2 ──
  Predicted actions  : ['Increase long-acting insulin from 24L to 28L for dawn phenomenon (fasting 155-18']
  Ground truth       : ['Increase long-acting insulin (Lantus/Basaglar) from 24U to 26U, or split the dos', 'Utilize extended/dual-wave bolus for dinner (50:50 split over 3-4 hours) to cove']
  Priority dist      : ['high']

── Sample 3 ──
  Predicted actions  : ['Discontinue any sulfonylurea or meglitinide immediately. If on metformin, add a ', 'Add a scheduled 15g complex carb snack at 15:00 daily to preempt the predictable']
  Ground truth       : ['Switch to low-GI breakfast (e.g., steel-cut oats with nuts, Greek yogurt with be', 'Prescribe a mandatory 15-20g complex carbohydrate snack at 14:30 to prevent the ']
  Priority dist      : ['high', 'medium']


## 8 · Save adapter

In [12]:
# ── Save LoRA adapter only (not merged) ───────────────────────────
# This is the smallest artifact (~80 MB vs 6 GB merged).
# At inference time you load base model + adapter separately.

adapter_path = f"{OUTPUT_DIR}/final-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✅ Adapter saved → {adapter_path}")

# ── List saved files ──────────────────────────────────────────────
import os
for f in sorted(os.listdir(adapter_path)):
    size = os.path.getsize(f"{adapter_path}/{f}") / 1024 / 1024
    print(f"  {f:<40} {size:.1f} MB")

[transformers] Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/diabetic-ai-lora/final-adapter/tokenizer_config.json.


✅ Adapter saved → /kaggle/working/diabetic-ai-lora/final-adapter
  README.md                                0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                114.3 MB
  chat_template.jinja                      0.0 MB
  tokenizer.json                           10.9 MB
  tokenizer_config.json                    0.0 MB


In [13]:
# ── Save merged model (optional — needed for GGUF / Ollama) ───────
# Uncomment if you want a standalone model file.
# Requires ~12 GB disk space.

# merged_path = f"{OUTPUT_DIR}/merged-model"
# model.save_pretrained_merged(
#     merged_path,
#     tokenizer,
#     save_method = "merged_16bit",
# )
# print(f"✅ Merged model saved → {merged_path}")

In [14]:
# ── Push to Hugging Face Hub (optional) ───────────────────────────
# Set HUB_REPO at the top of this notebook to enable.
# Requires: from huggingface_hub import login; login(token="hf_...")

if HUB_REPO:
    model.push_to_hub(HUB_REPO, token=True)
    tokenizer.push_to_hub(HUB_REPO, token=True)
    print(f"✅ Pushed to Hub → https://huggingface.co/{HUB_REPO}")
else:
    print("Hub push skipped (HUB_REPO not set)")

Hub push skipped (HUB_REPO not set)


## Summary

| Step | Status |
|------|--------|
| Model loaded (Qwen2.5-3B-Instruct) | ✅ |
| LoRA adapters attached (r=16, α=32) | ✅ |
| Dataset formatted with chat template | ✅ |
| SFTTrainer configured (eff. batch=32) | ✅ |
| Training (3 epochs, best ckpt saved) | ✅ |
| Evaluation (JSON validity + fields) | ✅ |
| Adapter saved to /kaggle/working | ✅ |

**Next step:** run inference from your app by loading base model + adapter.